# ThermoML to FAIRFluids Conversion Test

This notebook demonstrates the updated conversion that creates separate Fluid blocks for each PureOrMixtureData element.


In [1]:
from fairfluids.ThermoMLMapping.thermoml_mapper import ThermoMLMapper

# Initialize mapper
mapper = ThermoMLMapper()

# Convert the XML file
fairfluids_doc = mapper.convert_file('/home/sga/Code/FAIRFluids/fairfluids/data/thermoml_xml/spera_et_al_fpe_592_2025_114324.xml')

print(f"Document version: {fairfluids_doc.version.versionMajor}.{fairfluids_doc.version.versionMinor}")
print(f"Number of compounds: {len(fairfluids_doc.compound)}")
print(f"Number of fluids: {len(fairfluids_doc.fluid)}")

# Show first fluid with corrected mapping
if fairfluids_doc.fluid:
    fluid = fairfluids_doc.fluid[0]
    print(f"\nFirst Fluid Analysis:")
    print(f"  Compounds: {fluid.compounds}")
    print(f"  Properties: {len(fluid.property)}")
    print(f"  Parameters: {len(fluid.parameter)}")
    print(f"  Measurements: {len(fluid.measurement)}")
    
    print(f"\nProperties:")
    for i, prop in enumerate(fluid.property, 1):
        prop_type = prop.properties.value if prop.properties else "Unknown"
        print(f"  {i}. {prop.propertyID}: {prop_type}")
        if prop.unit:
            print(f"     Unit: {prop.unit.name}")
    
    print(f"\nParameters:")
    for i, param in enumerate(fluid.parameter, 1):
        param_type = param.parameter.value if param.parameter else "Unknown"
        associated = f" (associated: {param.associated_compounds})" if param.associated_compounds else ""
        print(f"  {i}. {param.parameterID}: {param_type}{associated}")
    
    print(f"\nSample Measurements:")
    for i, measurement in enumerate(fluid.measurement[:3], 1):
        print(f"  {i}. {measurement.measurement_id}")
        print(f"     Property values: {len(measurement.propertyValue)}")
        print(f"     Parameter values: {len(measurement.parameterValue)}")
        
        if measurement.propertyValue:
            for pv in measurement.propertyValue:
                print(f"       Property: {pv.propertyID} = {pv.propValue}")
        
        if measurement.parameterValue:
            for pv in measurement.parameterValue:
                print(f"       Parameter: {pv.parameterID} = {pv.paramValue}")


=== Filtering Fluid with 1 compounds and 5 measurements ===
✅ Keeping compound: compound_water (has non-zero mole fractions)
Final result: 1 compounds, 3 parameters
=== Filtering Fluid with 1 compounds and 5 measurements ===
✅ Keeping compound: compound_water (has non-zero mole fractions)
Final result: 1 compounds, 3 parameters
=== Filtering Fluid with 4 compounds and 5 measurements ===
✅ Keeping compound: compound_water (has non-zero mole fractions)
✅ Keeping compound: compound_choline (has non-zero mole fractions)
✅ Keeping compound: compound_16887006 (has non-zero mole fractions)
✅ Keeping compound: compound_glycerol (has non-zero mole fractions)
Final result: 4 compounds, 6 parameters
=== Filtering Fluid with 4 compounds and 5 measurements ===
✅ Keeping compound: compound_water (has non-zero mole fractions)
✅ Keeping compound: compound_choline (has non-zero mole fractions)
✅ Keeping compound: compound_16887006 (has non-zero mole fractions)
✅ Keeping compound: compound_glycerol (has

In [ ]:
# Show first fluid with detailed parameter info
if fairfluids_doc.fluid:
    fluid = fairfluids_doc.fluid[0]
    print(f"First Fluid Analysis:")
    print(f"  Compounds: {fluid.compounds}")
    print(f"  Properties: {len(fluid.property)}")
    print(f"  Parameters: {len(fluid.parameter)}")
    print(f"  Measurements: {len(fluid.measurement)}")
    
    print(f"\nParameters:")
    for i, param in enumerate(fluid.parameter, 1):
        param_type = param.parameter.value if param.parameter else "Unknown"
        associated = f" (associated: {param.associated_compounds})" if param.associated_compounds else ""
        print(f"  {i}. {param.parameterID}: {param_type}{associated}")
    
    print(f"\nSample Measurements:")
    for i, measurement in enumerate(fluid.measurement[:3], 1):
        print(f"  {i}. {measurement.measurement_id}")
        print(f"     Property values: {len(measurement.propertyValue)}")
        print(f"     Parameter values: {len(measurement.parameterValue)}")
        
        if measurement.propertyValue:
            for pv in measurement.propertyValue:
                print(f"       Property: {pv.propertyID} = {pv.propValue}")
        
        if measurement.parameterValue:
            for pv in measurement.parameterValue:
                print(f"       Parameter: {pv.parameterID} = {pv.paramValue}")


First Fluid Analysis:
  Compounds: ['compound_water']
  Properties: 1
  Parameters: 3
  Measurements: 5

Parameters:
  1. parameter_mole_fraction_water: Mole fraction (associated: compound_water)
  2. parameter_pressure_kPa: Pressure, kPa
  3. parameter_temperature: Temperature, K

Sample Measurements:
  1. measurement_1
     Property values: 1
     Parameter values: 3
       Property: density = 989.26692
       Parameter: parameter_temperature = 313.15
       Parameter: parameter_mole_fraction_water = 1.0
       Parameter: parameter_pressure_kPa = 100.0
  2. measurement_2
     Property values: 1
     Parameter values: 3
       Property: density = 984.23857
       Parameter: parameter_temperature = 323.15
       Parameter: parameter_mole_fraction_water = 1.0
       Parameter: parameter_pressure_kPa = 100.0
  3. measurement_3
     Property values: 1
     Parameter values: 3
       Property: density = 978.73098
       Parameter: parameter_temperature = 333.15
       Parameter: parameter_

In [5]:
# Scan fluids for newly mapped properties and show samples
from collections import defaultdict

wanted = {
    "molarEnthalpy",
    "isobaricExpansionCoefficient",
    "excessMolarVolume",
    "excessMolarEnthalpy",
    "diffusionCoefficient",
    "henrysLawConstant",
}

prop_counts = defaultdict(int)
samples = {k: [] for k in wanted}

for fluid in fairfluids_doc.fluid:
    # count properties
    for p in fluid.property:
        if p.properties:
            key = p.properties.value
            if key in wanted:
                prop_counts[key] += 1
                # capture up to 3 samples with unit
                if len(samples[key]) < 3:
                    samples[key].append((p.propertyID, p.unit.name if p.unit else None))

print("Discovered properties (counts):")
for k in sorted(wanted):
    print(f"  {k}: {prop_counts[k]}")

print("\nSamples (propertyID, unit):")
for k in sorted(wanted):
    if samples[k]:
        print(f"  {k}:")
        for pid, unit in samples[k]:
            print(f"    - {pid}, unit={unit}")

# Show first fluid that has any of the wanted properties and its first measurement linking IDs
for fluid in fairfluids_doc.fluid:
    has_wanted = any((p.properties and p.properties.value in wanted) for p in fluid.property)
    if has_wanted:
        print("\nFluid with target properties:")
        print("  Compounds:", fluid.compounds)
        print("  Properties:")
        for p in fluid.property:
            if p.properties and p.properties.value in wanted:
                print(f"    - {p.propertyID}: {p.properties.value} (unit={p.unit.name if p.unit else None})")
        # show first measurement mapping IDs
        if fluid.measurement:
            m = fluid.measurement[0]
            print(f"  Measurement {m.measurement_id}:")
            for pv in m.propertyValue:
                print(f"    propertyValue -> {pv.propertyID} = {pv.propValue}")
            for vv in m.parameterValue:
                print(f"    parameterValue -> {vv.parameterID} = {vv.paramValue}")
        break


Discovered properties (counts):
  diffusionCoefficient: 28
  excessMolarEnthalpy: 8
  excessMolarVolume: 8
  henrysLawConstant: 16
  isobaricExpansionCoefficient: 8
  molarEnthalpy: 8

Samples (propertyID, unit):
  diffusionCoefficient:
    - diffusionCoefficient, unit=Self diffusion coefficient, m2/s
    - diffusionCoefficient, unit=Self diffusion coefficient, m2/s
    - diffusionCoefficient, unit=Self diffusion coefficient, m2/s
  excessMolarEnthalpy:
    - excessMolarEnthalpy, unit=Excess molar enthalpy (molar enthalpy of mixing), kJ/mol
    - excessMolarEnthalpy, unit=Excess molar enthalpy (molar enthalpy of mixing), kJ/mol
    - excessMolarEnthalpy, unit=Excess molar enthalpy (molar enthalpy of mixing), kJ/mol
  excessMolarVolume:
    - excessMolarVolume, unit=Excess molar volume, m3/mol
    - excessMolarVolume, unit=Excess molar volume, m3/mol
    - excessMolarVolume, unit=Excess molar volume, m3/mol
  henrysLawConstant:
    - henrysLawConstant, unit=Henry's Law constant (amount 

In [6]:
with open("fairfluids_output.json", "w") as f:
    f.write(fairfluids_doc.model_dump_json(indent=2))